# So sanh TradingView va SQL: phat hien nen lich su bi sua

Notebook chi-doc: dung lai dung auth/WebSocket/SQL access cua `dp_program` engine de tai
500 nen dong gan nhat tren TradingView cho toan bo pair live hien hanh, doi chieu voi
500 nen gan nhat dang luu trong `DWH.Fact_OHLCV`. Chi so 4 gia `open/high/low/close`, bo
`volume`.

Muc tieu: tim nen ma SQL da luu khac voi gia TradingView dang tra ve hien tai o cung
`BarTime` -- dau hieu broker co the da sua du lieu lich su sau khi DP da capture.
Notebook khong goi Redis, khong ghi Fact, khong backfill. Chay tuan tu Run All la ra
bao cao, khong dung ipywidgets.

In [ ]:
from __future__ import annotations

from collections import defaultdict
from datetime import datetime, timezone
from decimal import Decimal
from pathlib import Path
from time import perf_counter
import sys

import pandas as pd
from IPython.display import display

sys.dont_write_bytecode = True
candidates = [Path.cwd(), Path.cwd() / 'core_program', *Path.cwd().parents]
REPOSITORY = next((path for path in candidates if (path / 'src' / 'dp_program').is_dir()), None)
if REPOSITORY is None:
    raise RuntimeError('Open this notebook from dp_program_v3, core_program, or research.')

sys.path.insert(0, str(REPOSITORY / 'src'))

from dp_program.configuration import load_config
from dp_program.engine.auth import _best_material, token_seconds_remaining
from dp_program.engine.pipeline import validate_candles
from dp_program.engine.sql_connector import read_latest_candles_for_pairs, select_pairs
from dp_program.engine.websocket import FetchRequest, _fetch_batch_once, request_key

BARS = 500
FIELDS = ('open', 'high', 'low', 'close')
ALLOWED_LIVE_LAG_BARS = 1

config = load_config(REPOSITORY.parent / 'run_dp' / 'config.yaml')

pairs = select_pairs(config, live=True)
pair_keys = [(int(symbol['symbol_id']), timeframe['code']) for symbol, timeframe in pairs]
pair_meta = {
    (int(symbol['symbol_id']), timeframe['code']): {
        'symbol': f"{symbol['exchange']}:{symbol['symbol']}",
        'raw_symbol': symbol['symbol'],
        'timeframe': timeframe['code'],
    }
    for symbol, timeframe in pairs
}


def elapsed(started):
    return f'{perf_counter() - started:.2f}s'


def as_time(value):
    if isinstance(value, datetime):
        return (value.astimezone(timezone.utc) if value.tzinfo else value).replace(tzinfo=None, microsecond=0)
    text = str(value or '').strip()
    if text.isdigit():
        return datetime.fromtimestamp(int(float(text)), timezone.utc).replace(tzinfo=None)
    return datetime.strptime(text, '%Y-%m-%d %H:%M:%S')


def as_ohlc(values):
    return tuple(Decimal(str(value)) for value in values)


print(f'Live pairs: {len(pairs)} | TradingView reference candles per pair: {BARS}')

In [ ]:
started = perf_counter()
_cache, auth = _best_material(config)
if auth is None or token_seconds_remaining(auth[1]) <= 300:
    raise RuntimeError('TradingView token is unavailable or expires too soon. Run normal DP authentication first.')

tv_config = dict(config['tradingview'], auth_token=auth[1], cookie=auth[2])
now = datetime.now(timezone.utc)
by_symbol = defaultdict(list)
for symbol, timeframe in pairs:
    by_symbol[(symbol['exchange'], symbol['symbol'])].append((symbol, timeframe))

tv_data = {}
tv_batches = []
for (exchange, symbol_name), group in by_symbol.items():
    batch_started = perf_counter()
    if len(group) > 15:
        raise RuntimeError('This notebook supports at most 15 timeframes per symbol.')
    requests = [FetchRequest(symbol, timeframe, BARS + 1, BARS + 1) for symbol, timeframe in group]
    fetched, metrics = _fetch_batch_once(tv_config, requests, config['backfill']['max_bars_per_request'])
    for request in requests:
        closed = validate_candles(
            fetched[request_key(request)].candles, request.timeframe, closed_only=True, now=now
        )[-BARS:]
        tv_data[request_key(request)] = {
            as_time(candle['timestamp']): as_ohlc(candle[field] for field in FIELDS)
            for candle in closed
        }
    tv_batches.append({
        'symbol': f'{exchange}:{symbol_name}',
        'series': len(requests),
        'seconds': round(perf_counter() - batch_started, 3),
        'connect_seconds': metrics.get('connect_seconds'),
        'received_bytes': metrics.get('received_bytes'),
    })

tv_batch_frame = pd.DataFrame(tv_batches)
print(f'TradingView loaded: {len(tv_data)}/{len(pairs)} pairs in {elapsed(started)}')
display(tv_batch_frame)

In [ ]:
started = perf_counter()
sql_rows = read_latest_candles_for_pairs(config, pair_keys, BARS)
sql_data = {
    key: {as_time(row[0]): as_ohlc(row[1:5]) for row in rows}
    for key, rows in sql_rows.items()
}
sql_frame = pd.DataFrame([
    {'symbol': pair_meta[key]['symbol'], 'timeframe': key[1], 'bars': len(sql_data[key])}
    for key in pair_keys
])

print(f'SQL loaded: {len(sql_data)}/{len(pairs)} pairs in {elapsed(started)}')
display(sql_frame.head(20))

In [ ]:
def pct_from_tv(tv_value, target_value):
    if tv_value == 0:
        return None
    return float((target_value - tv_value) / abs(tv_value) * Decimal('100'))


def pct_stats(values):
    numbers = pd.to_numeric(pd.Series(list(values)), errors='coerce').dropna()
    if numbers.empty:
        return {
            'min_abs_diff_pct': None, 'max_abs_diff_pct': None,
            'avg_abs_diff_pct': None, 'median_abs_diff_pct': None,
            'p95_abs_diff_pct': None,
        }
    return {
        'min_abs_diff_pct': round(float(numbers.min()), 6),
        'max_abs_diff_pct': round(float(numbers.max()), 6),
        'avg_abs_diff_pct': round(float(numbers.mean()), 6),
        'median_abs_diff_pct': round(float(numbers.median()), 6),
        'p95_abs_diff_pct': round(float(numbers.quantile(0.95)), 6),
    }


PAIR_COLUMNS = [
    'pair', 'symbol', 'timeframe', 'status', 'tv_bars', 'sql_bars',
    'allowed_lag_bars', 'missing_error_bars', 'extra_window_bars',
    'revision_bars', 'error_bars', 'error_bar_pct', 'field_diff_count',
    'min_abs_diff_pct', 'max_abs_diff_pct', 'avg_abs_diff_pct',
    'median_abs_diff_pct', 'p95_abs_diff_pct',
    'worst_bartime', 'worst_field', 'worst_tv_value', 'worst_sql_value', 'worst_diff_pct',
]
DIFF_COLUMNS = [
    'pair', 'symbol', 'timeframe', 'bartime', 'issue', 'field',
    'tv_value', 'sql_value', 'diff', 'diff_pct', 'abs_diff_pct',
]

pair_rows, diff_rows = [], []

for key in pair_keys:
    meta = pair_meta[key]
    label = f"{meta['symbol']}/{meta['timeframe']}"
    tv = tv_data.get(key, {})
    sql = sql_data.get(key, {})
    tv_times, sql_times = set(tv), set(sql)
    missing_bartimes = sorted(tv_times - sql_times)
    extra_bartimes = sorted(sql_times - tv_times)
    field_diff_rows, revision_bars = [], 0

    for bartime in sorted(tv_times & sql_times):
        bar_has_diff = False
        for index, field in enumerate(FIELDS):
            tv_value = tv[bartime][index]
            sql_value = sql[bartime][index]
            if tv_value == sql_value:
                continue
            bar_has_diff = True
            percent = pct_from_tv(tv_value, sql_value)
            field_diff_rows.append({
                'pair': label, 'symbol': meta['symbol'], 'timeframe': meta['timeframe'],
                'bartime': bartime, 'issue': 'revision_candidate', 'field': field,
                'tv_value': tv_value, 'sql_value': sql_value,
                'diff': sql_value - tv_value,
                'diff_pct': percent,
                'abs_diff_pct': abs(percent) if percent is not None else None,
            })
        revision_bars += int(bar_has_diff)

    allowed_lag = (
        len(missing_bartimes) == ALLOWED_LIVE_LAG_BARS
        and revision_bars == 0
        and bool(tv_times)
        and missing_bartimes[-1] == max(tv_times)
        and len(extra_bartimes) <= ALLOWED_LIVE_LAG_BARS
        and all(extra < min(tv_times) for extra in extra_bartimes)
    )
    coverage_issue = 'allowed_one_bar_lag' if allowed_lag else 'missing_from_sql'
    extra_issue = 'allowed_one_bar_lag' if allowed_lag else 'extra_outside_tv_window'

    for bartime in missing_bartimes:
        diff_rows.append({
            'pair': label, 'symbol': meta['symbol'], 'timeframe': meta['timeframe'],
            'bartime': bartime, 'issue': coverage_issue, 'field': 'timestamp',
            'tv_value': bartime, 'sql_value': None, 'diff': None, 'diff_pct': None, 'abs_diff_pct': None,
        })
    for bartime in extra_bartimes:
        diff_rows.append({
            'pair': label, 'symbol': meta['symbol'], 'timeframe': meta['timeframe'],
            'bartime': bartime, 'issue': extra_issue, 'field': 'timestamp',
            'tv_value': None, 'sql_value': bartime, 'diff': None, 'diff_pct': None, 'abs_diff_pct': None,
        })
    diff_rows.extend(field_diff_rows)

    allowed_lag_bars = len(missing_bartimes) if allowed_lag else 0
    missing_error_bars = 0 if allowed_lag else len(missing_bartimes)
    error_bars = missing_error_bars + revision_bars
    stat = pct_stats(row['abs_diff_pct'] for row in field_diff_rows)
    worst = max(field_diff_rows, key=lambda row: row['abs_diff_pct'] or -1, default={})
    pair_rows.append({
        'pair': label, 'symbol': meta['symbol'], 'timeframe': meta['timeframe'],
        'status': 'ERROR' if error_bars else ('ALLOWED_LAG' if allowed_lag_bars else 'OK'),
        'tv_bars': len(tv), 'sql_bars': len(sql),
        'allowed_lag_bars': allowed_lag_bars,
        'missing_error_bars': missing_error_bars,
        'extra_window_bars': len(extra_bartimes),
        'revision_bars': revision_bars,
        'error_bars': error_bars,
        'error_bar_pct': round(100 * error_bars / len(tv), 4) if tv else 0,
        'field_diff_count': len(field_diff_rows),
        **stat,
        'worst_bartime': worst.get('bartime'),
        'worst_field': worst.get('field'),
        'worst_tv_value': worst.get('tv_value'),
        'worst_sql_value': worst.get('sql_value'),
        'worst_diff_pct': worst.get('diff_pct'),
    })

pair_stats_frame = pd.DataFrame(pair_rows, columns=PAIR_COLUMNS)
diff_frame = pd.DataFrame(diff_rows, columns=DIFF_COLUMNS)

print(f'Pairs checked: {len(pair_stats_frame)} | TradingView reference candles per pair: {BARS}')

In [ ]:
tv_bars_total = int(pair_stats_frame['tv_bars'].sum())
error_bars_total = int(pair_stats_frame['error_bars'].sum())
overview = pd.DataFrame([{
    'pairs_checked': len(pair_stats_frame),
    'clean_pairs': int(pair_stats_frame['status'].eq('OK').sum()),
    'pairs_with_allowed_lag': int(pair_stats_frame['allowed_lag_bars'].gt(0).sum()),
    'pairs_with_revision': int(pair_stats_frame['revision_bars'].gt(0).sum()),
    'tv_bars': tv_bars_total,
    'allowed_lag_bars': int(pair_stats_frame['allowed_lag_bars'].sum()),
    'missing_error_bars': int(pair_stats_frame['missing_error_bars'].sum()),
    'revision_bars': int(pair_stats_frame['revision_bars'].sum()),
    'error_bars': error_bars_total,
    'error_bar_pct': round(100 * error_bars_total / tv_bars_total, 4) if tv_bars_total else 0,
}])

revision_detail = diff_frame[diff_frame['issue'] == 'revision_candidate'].sort_values(
    ['abs_diff_pct'], ascending=False, na_position='last'
)
worst_pairs = pair_stats_frame[pair_stats_frame['error_bars'].gt(0)].sort_values(
    ['error_bar_pct', 'max_abs_diff_pct'], ascending=[False, False], na_position='last'
).head(30)
field_stats_rows = []
if not revision_detail.empty:
    for field, rows in revision_detail.groupby('field', sort=False):
        field_stats_rows.append({'field': field, 'revision_rows': len(rows), **pct_stats(rows['abs_diff_pct'])})
field_stats_frame = pd.DataFrame(field_stats_rows)

print('=== Overview ===')
display(overview)
print('=== Worst pairs (theo % bar loi) ===')
display(worst_pairs)
print('=== Lech OHLC theo field (trong cac bar nghi bi sua) ===')
display(field_stats_frame)
print(f'=== Tat ca revision candidate ({len(revision_detail)} dong), sap theo abs_diff_pct giam dan ===')
display(revision_detail.head(100))

In [ ]:
def query_pair_stats(symbol=None, timeframe=None, status=None, limit=50):
    rows = pair_stats_frame.copy()
    if symbol:
        rows = rows[rows['symbol'].str.contains(symbol, case=False, na=False)]
    if timeframe:
        rows = rows[rows['timeframe'].eq(timeframe)]
    if status:
        rows = rows[rows['status'].eq(status)]
    return rows.sort_values(['error_bar_pct', 'max_abs_diff_pct'], ascending=[False, False], na_position='last').head(limit)


def query_diffs(symbol=None, timeframe=None, issue=None, field=None, limit=50):
    rows = diff_frame.copy()
    if rows.empty:
        return rows
    if symbol:
        rows = rows[rows['symbol'].str.contains(symbol, case=False, na=False)]
    if timeframe:
        rows = rows[rows['timeframe'].eq(timeframe)]
    if issue:
        rows = rows[rows['issue'].eq(issue)]
    if field:
        rows = rows[rows['field'].eq(field)]
    return rows.sort_values(['abs_diff_pct', 'bartime'], ascending=[False, False], na_position='last').head(limit)

## Cach doc ket qua

- `overview`: tong quan toan bo pair live -- so pair sach, so pair chi lag 1 bar duoc
  phep, so pair co nen nghi bi sua (`pairs_with_revision`), tong bar loi va ty trong.
- `worst_pairs`: cac pair loi nang nhat, sap theo `error_bar_pct` roi `max_abs_diff_pct`.
- `field_stats_frame`: muc lech theo tung field `open/high/low/close` trong cac bar
  nghi bi sua.
- `revision_detail`: chi tiet tung bar/field ma SQL da luu khac voi gia TradingView
  dang tra ve hien tai o cung `BarTime` -- day la danh sach chinh de xem broker co sua
  du lieu lich su hay khong. `diff = sql_value - tv_value`.
- `allowed_one_bar_lag`: SQL chi thieu dung nen moi nhat cua TradingView do do tre chu
  ky live, khong tinh la loi.
- `missing_from_sql`: TradingView co `BarTime` do nhung SQL khong co trong so nen doc
  ra, va khong thuoc truong hop lag 1 bar -- dang chu y, co the la gap that.
- `extra_outside_tv_window`: SQL co `BarTime` cu hon nen co nhat trong so nen
  TradingView hien tai -- binh thuong, chi vi TradingView gioi han cua so, khong phai
  loi.
- **Gioi han cua cach so 2 nguon nay**: notebook chi ket luan "SQL khac TradingView
  hien tai", chua tu chung minh chieu nguyen nhan (broker sua sau khi capture, hay DP
  capture sai tu dau). Muon chac hon, doi chieu them `Fact_OHLCV.CreatedAt` cua cac bar
  trong `revision_detail` xem da capture tu bao lau truoc.
- Dung `query_pair_stats(symbol='BTCUSD')` hoac
  `query_diffs(symbol='BTCUSD', timeframe='H1', issue='revision_candidate', limit=100)`
  de loc thu cong.
- Notebook chi doc TradingView va SQL, khong ghi Fact, khong goi Redis.